# Data Cleaning for Phases 3 and 4

**INFO 2950 | Insider Disclosures and Stock Returns**

**Group:** Vansh Kakra, Dev Shukla, and Wilson Wang.

In Phase 2, we collected the source data and developed the cleaning methods described in the [Phase 2 notebook](https://github.com/Vansh-Kakra/DSproject/blob/7f8759d9b4af42a03e0c08f3da9c2baccd938f72/insider-trading-phase2/phase2_insider_trading.ipynb). Here, we use those saved inputs and methods to prepare the datasets for Phase 3 analysis.

The notebook follows the processing order from start to finish. We begin by loading and inspecting the source files, then prepare the membership history and clean the transactions. After adding insider roles and matching index membership, we calculate weekly returns and combine trades into disclosure events. The final steps calculate abnormal returns, check the exports, and save the CSV files. Each section explains the code that follows it.

### Running the notebook

This notebook uses the same Python environment and required packages as Phase 2. The saved raw files are under `insider-trading-phase2/data/raw`, including the 24 SEC quarter folders, historical membership CSV, and daily prices. To reproduce the results, open the notebook from the repository root or its own folder, then **restart the kernel and run all cells in order**. The price data has already been collected, so there is no need to rerun `collect_prices.py`.

Raw input paths in Section 1 are relative to `insider-trading-phase2`. All new exports go under **`insider-trading-phase3-4`**. The submitted Phase 2 files remain unchanged.

## 1. Source Data and Setup

### Previously Collected Source Files

**SEC disclosures:** The project obtained 24 quarterly ZIP archives covering **2020q1 through 2025q4** from the [SEC Insider Transactions Data Sets](https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets). Extracted files are stored in `data/raw/sec/YYYYqQ_form345/`. We read `SUBMISSION.tsv`, `NONDERIV_TRANS.tsv`, `REPORTINGOWNER.tsv`, and `FOOTNOTES.tsv` from each quarter, preserving a quarter identifier. Other tables in the archives are retained as raw material but are not part of the analysis. The original SEC retrieval date was not recorded.

**Index membership:** The supplied `data/raw/historical/S&P 500 Historical Components & Changes (Updated).csv` comes from the community-maintained [fja05680/sp500 repository](https://github.com/fja05680/sp500). Dated ticker lists allow us to select companies that belonged to the index when filings occurred, including later departures. The original retrieval date is not recorded. We preserve the raw CSV and use its snapshots to create continuous membership intervals.

**Prices:** `collect_prices.py` downloaded daily prices from Yahoo Finance using yfinance on **October 19, 2026**, requesting data from January 1, 2019 onward for 627 historical index symbols plus SPY. The saved observations span January 2, 2019 through October 19, 2026. The extra pre-2020 history supports market-model estimation, and later prices support follow-up after 2025 filings. The request used `auto_adjust=True` to account for splits and dividends. Failed symbols were retried; unusable results were recorded in `data/raw/prices/daily_stock_prices_missing.csv`. The main file is `data/raw/prices/daily_stock_prices.csv.gz`; the earlier `weekly_stock_prices.csv` remains for reference and is not the input for the weekly-return calculations below. There are 552 stock tickers with data plus SPY (553 symbols in total) and 75 missing symbols.

We use the saved files throughout the notebook. The collection code remains in `collect_prices.py`. We save the intermediate and processed datasets separately.

### How the Sources Fit Together

The three sources serve different purposes. **SEC filings** identify insider transactions and their filing dates. **Historical S&P 500 membership** tells us whether each company belonged to the index on its filing date. **Yahoo Finance prices** provide the stock returns, with SPY serving as the market benchmark.

These inputs were saved in the [raw-data folder](https://github.com/Vansh-Kakra/DSproject/tree/7f8759d9b4af42a03e0c08f3da9c2baccd938f72/insider-trading-phase2/data/raw) during Phase 2. We start by loading the libraries and locating those files.

### 1.1 Loading the Libraries

First, we import the libraries needed to read tables, combine records, and calculate returns. The plotting settings are carried over from the Phase 2 environment.

In [1]:
# Imports and plot settings
import numpy as np
import pandas as pd
from scipy import stats
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1.2 Setting Input and Output Locations

The next cell locates the repository and the saved Phase 2 raw-data folder. It checks that the input folder exists, then creates `data/interim` and `data/processed` under `insider-trading-phase3-4`. These separate output locations let us reuse the original inputs without overwriting the Phase 2 files.

In [2]:
# Locate the original inputs and isolate all new exports from the submitted Phase 2 files.
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "insider-trading-phase2").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the repository root or insider-trading-phase3-4 folder.")
raw_dir = repo_root / "insider-trading-phase2" / "data" / "raw"
if not raw_dir.is_dir():
    raise FileNotFoundError(f"Restore the saved Phase 2 raw data at {raw_dir} before running this notebook.")
project_dir = repo_root / "insider-trading-phase3-4"
sec_dir = raw_dir / "sec"
interim_dir = project_dir / "data" / "interim"
processed_dir = project_dir / "data" / "processed"
for output_dir in [interim_dir, processed_dir]:
    output_dir.mkdir(parents=True, exist_ok=True)
print("Read inputs:", raw_dir)
print("Write outputs:", project_dir / "data")

Read inputs: C:\Users\vansh\.codex\.chatgpt-projects\g-p-6ad697e67f448191ba13f713a97c3142\step4-run\DSproject-phase2-part1-sec-cleaning\insider-trading-phase2\data\raw
Write outputs: C:\Users\vansh\.codex\.chatgpt-projects\g-p-6ad697e67f448191ba13f713a97c3142\step4-run\DSproject-phase2-part1-sec-cleaning\insider-trading-phase3-4\data


### 1.3 Loading the SEC Records

With the file locations set, we load SEC insider-trading records from 24 quarterly datasets covering 2020 Q1 through 2025 Q4. For each quarter, we read four required files: `SUBMISSION.tsv`, `NONDERIV_TRANS.tsv`, `REPORTINGOWNER.tsv`, and `FOOTNOTES.tsv`. Before reading the files, we verify that all expected files are present. This check stops the run if any quarterly input is missing. We initially read all fields as text to preserve identifiers such as accession numbers and CIKs, including their leading zeros. Before combining the quarterly tables, we add a `SOURCE_QUARTER` field to each record so that we can trace it to its original SEC release. The combined tables provide filing-level information, transaction-level details, filer relationships, and associated footnotes for the full study period.

In [3]:
# Load the four SEC tables from all 24 quarters, preserving their source quarter.
from pathlib import Path
from time import perf_counter

quarter_folders = [
    sec_dir / f"{year}q{quarter}_form345"
    for year in range(2020, 2026)
    for quarter in range(1, 5)
]
table_names = ["SUBMISSION", "NONDERIV_TRANS", "REPORTINGOWNER", "FOOTNOTES"]
missing_files = [str(folder / f"{table}.tsv")
                 for folder in quarter_folders for table in table_names
                 if not (folder / f"{table}.tsv").is_file()]
if missing_files:
    raise FileNotFoundError("Check the SEC folder location or missing files:\n" + "\n".join(missing_files))

sec_tables = {}
file_counts = []
load_start = perf_counter()
for table in table_names:
    quarter_frames = []
    for folder in quarter_folders:
        frame = pd.read_csv(folder / f"{table}.tsv", sep="\t", dtype="string")
        frame["SOURCE_QUARTER"] = folder.name.split("_")[0]
        file_counts.append({"quarter": frame["SOURCE_QUARTER"].iloc[0] if len(frame) else folder.name.split("_")[0],
                            "table": table, "rows": len(frame)})
        quarter_frames.append(frame)
    sec_tables[table] = pd.concat(quarter_frames, ignore_index=True)
    del quarter_frames, frame
    print(f"{table}: {len(sec_tables[table]):,} rows loaded", flush=True)

df_submission = sec_tables["SUBMISSION"]
df_transactions = sec_tables["NONDERIV_TRANS"]
df_owners = sec_tables["REPORTINGOWNER"]
df_footnotes = sec_tables["FOOTNOTES"]
df_file_counts = pd.DataFrame(file_counts)
print(f"Loaded {len(file_counts)} files from {len(quarter_folders)} quarters in {perf_counter() - load_start:.1f} seconds.")

SUBMISSION: 1,211,888 rows loaded
NONDERIV_TRANS: 1,841,674 rows loaded
REPORTINGOWNER: 1,323,957 rows loaded
FOOTNOTES: 2,865,764 rows loaded
Loaded 96 files from 24 quarters in 25.4 seconds.


### 1.4 Checking the Source Tables

The quarterly SEC tables have now been loaded and combined. Before filtering any transactions, we inspect the number of rows, columns, and distinct source quarters in each table. The quarterly row counts help us spot missing coverage, unexpectedly small quarters, or unusual changes in filing volume.

We also inspect the transaction columns and a sample of raw rows. This lets us check that fields such as the transaction date, purchase/sale code, shares, price, ownership type, and accession number are available and formatted as expected.

In [4]:
# Inspect combined table sizes, source-quarter coverage, and raw transaction fields.
from IPython.display import display

summary = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": frame.shape[1],
     "quarters": frame["SOURCE_QUARTER"].nunique()}
    for name, frame in sec_tables.items()
])
display(summary)
display(df_file_counts.pivot(index="quarter", columns="table", values="rows"))
print("Transaction columns:", df_transactions.columns.tolist())
display(df_transactions.head())

Transaction columns: ['ACCESSION_NUMBER', 'NONDERIV_TRANS_SK', 'SECURITY_TITLE', 'SECURITY_TITLE_FN', 'TRANS_DATE', 'TRANS_DATE_FN', 'DEEMED_EXECUTION_DATE', 'DEEMED_EXECUTION_DATE_FN', 'TRANS_FORM_TYPE', 'TRANS_CODE', 'EQUITY_SWAP_INVOLVED', 'EQUITY_SWAP_TRANS_CD_FN', 'TRANS_TIMELINESS', 'TRANS_TIMELINESS_FN', 'TRANS_SHARES', 'TRANS_SHARES_FN', 'TRANS_PRICEPERSHARE', 'TRANS_PRICEPERSHARE_FN', 'TRANS_ACQUIRED_DISP_CD', 'TRANS_ACQUIRED_DISP_CD_FN', 'SHRS_OWND_FOLWNG_TRANS', 'SHRS_OWND_FOLWNG_TRANS_FN', 'VALU_OWND_FOLWNG_TRANS', 'VALU_OWND_FOLWNG_TRANS_FN', 'DIRECT_INDIRECT_OWNERSHIP', 'DIRECT_INDIRECT_OWNERSHIP_FN', 'NATURE_OF_OWNERSHIP', 'NATURE_OF_OWNERSHIP_FN', 'SOURCE_QUARTER']


,table,rows,columns,quarters
0,SUBMISSION,1211888,15,24
1,NONDERIV_TRANS,1841674,29,24
2,REPORTINGOWNER,1323957,14,24
3,FOOTNOTES,2865764,4,24


table,FOOTNOTES,NONDERIV_TRANS,REPORTINGOWNER,SUBMISSION
quarter,,,,
2020q1,159716,105852,73693,69408
2020q2,113675,70429,55817,51459
2020q3,92931,60619,43766,39096
2020q4,108077,72973,49045,43607
2021q1,188895,125620,81046,74944
2021q2,140976,89108,66036,58798
2021q3,111270,74002,50774,44057
2021q4,118799,80096,52858,46675
2022q1,171741,110327,76260,71395


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,SECURITY_TITLE,SECURITY_TITLE_FN,TRANS_DATE,TRANS_DATE_FN,DEEMED_EXECUTION_DATE,DEEMED_EXECUTION_DATE_FN,TRANS_FORM_TYPE,TRANS_CODE,EQUITY_SWAP_INVOLVED,EQUITY_SWAP_TRANS_CD_FN,TRANS_TIMELINESS,TRANS_TIMELINESS_FN,TRANS_SHARES,TRANS_SHARES_FN,TRANS_PRICEPERSHARE,TRANS_PRICEPERSHARE_FN,TRANS_ACQUIRED_DISP_CD,TRANS_ACQUIRED_DISP_CD_FN,SHRS_OWND_FOLWNG_TRANS,SHRS_OWND_FOLWNG_TRANS_FN,VALU_OWND_FOLWNG_TRANS,VALU_OWND_FOLWNG_TRANS_FN,DIRECT_INDIRECT_OWNERSHIP,DIRECT_INDIRECT_OWNERSHIP_FN,NATURE_OF_OWNERSHIP,NATURE_OF_OWNERSHIP_FN,SOURCE_QUARTER
0,0001446472-20-000002,281943,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,P,0,<NA>,<NA>,<NA>,7800.0,<NA>,2.88,F1,A,<NA>,1537800.0,F2,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
1,0001084991-20-000014,379212,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,M,0,<NA>,<NA>,<NA>,1302.0,<NA>,0.0,F1,A,<NA>,8560.0,<NA>,<NA>,<NA>,I,<NA>,By Rabbi Trust,<NA>,2020q1
2,0001104659-20-041411,245379,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,0,F1,<NA>,<NA>,1037.0,<NA>,37.76,<NA>,D,<NA>,54890.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
3,0001104659-20-041411,245380,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,F,0,F2,<NA>,<NA>,552.0,<NA>,40.74,<NA>,D,<NA>,54338.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
4,0001104659-20-041409,272457,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,0,F1,<NA>,<NA>,953.0,<NA>,37.76,<NA>,D,<NA>,24497.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1


### 1.5 Loading the Membership History

Next, we load the historical S&P 500 membership CSV. Each row contains a snapshot date and a comma-separated list of ticker symbols. The code parses the dates, reports the coverage and missing values, and displays the first few rows. We will use these snapshots to construct membership intervals in the next section.

In [5]:
# Load and inspect the supplied historical membership CSV.
from pathlib import Path

historical_file = raw_dir / "historical" / "S&P 500 Historical Components & Changes (Updated).csv"
df_historical = pd.read_csv(historical_file)
historical_dates = pd.to_datetime(df_historical["date"], format="%Y-%m-%d")
print(f"Historical membership: {len(df_historical):,} rows, {df_historical.shape[1]} columns")
print(f"Snapshot dates: {historical_dates.min().date()} to {historical_dates.max().date()}")
print("Missing values:", df_historical.isna().sum().to_dict())
display(df_historical.head())

Historical membership: 2,720 rows, 2 columns
Snapshot dates: 1996-01-02 to 2026-08-18
Missing values: {'date': 0, 'tickers': 0}


,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


## 2. Preparing Historical Membership

### 2.1 Cleaning Snapshot Dates and Tickers

We begin the membership cleaning by parsing snapshot dates, standardizing ticker spacing and capitalization, and removing repeated tickers and duplicate snapshots. The code stops if it finds missing values, empty ticker lists, or conflicting snapshots so that we can review them before continuing.

Using the SEC filing-date range, we keep the last snapshot before the study begins and all subsequent snapshots through its end. We treat each snapshot date as the start of its recorded membership, which assumes that the recorded snapshot dates reflect membership changes.

In [6]:
# Clean historical membership snapshots and retain coverage for the actual SEC filing window.
filing_dates = pd.to_datetime(df_submission["FILING_DATE"], format="mixed", errors="raise")
if filing_dates.isna().any() or filing_dates.empty:
    raise ValueError("Review missing SEC filing dates before defining the study window.")
study_start = filing_dates.min().normalize()
study_end = filing_dates.max().normalize()
study_end_exclusive = study_end + pd.Timedelta(days=1)

historical = df_historical[["date", "tickers"]].copy()
if historical.isna().any().any():
    raise ValueError("Review missing snapshot dates or ticker lists before cleaning.")
historical["date"] = pd.to_datetime(historical["date"], format="%Y-%m-%d", errors="raise")
ticker_lists = historical["tickers"].astype("string").str.split(",")
ticker_lists = ticker_lists.map(lambda values: [ticker.strip().upper() for ticker in values if ticker.strip()])
repeated_tickers = int(ticker_lists.map(lambda values: len(values) - len(set(values))).sum())
if ticker_lists.map(len).eq(0).any():
    raise ValueError("An empty membership list requires review.")
historical["tickers"] = ticker_lists.map(lambda values: ",".join(sorted(set(values))))
exact_duplicates = int(historical.duplicated(["date", "tickers"]).sum())
historical = historical.drop_duplicates(["date", "tickers"]).sort_values("date")
if historical["date"].duplicated().any():
    raise ValueError("Conflicting membership snapshots share the same date; review them first.")

baseline = historical.loc[historical["date"].le(study_start)].tail(1)
if baseline.empty:
    raise ValueError("A membership snapshot on or before the first filing is required.")
changes = historical.loc[historical["date"].gt(study_start) & historical["date"].le(study_end)]
historical_snapshots = pd.concat([baseline, changes], ignore_index=True)
historical_snapshots = historical_snapshots.rename(columns={"date": "snapshot_date"})
historical_snapshots["valid_from"] = historical_snapshots["snapshot_date"].clip(lower=study_start)
historical_snapshots["valid_to"] = historical_snapshots["snapshot_date"].shift(-1).fillna(study_end_exclusive)

print(f"SEC filing window: {study_start.date()} through {study_end.date()}")
print(f"Starting membership snapshot: {baseline['date'].iloc[0].date()}")
print(f"Original snapshots: {len(df_historical):,}; retained snapshots: {len(historical_snapshots):,}")
print(f"Exact duplicate snapshots removed: {exact_duplicates}; repeated ticker entries removed: {repeated_tickers}")
display(historical_snapshots[["snapshot_date", "valid_from", "valid_to"]].head())

SEC filing window: 2020-01-02 through 2025-12-31
Starting membership snapshot: 2019-12-23
Original snapshots: 2,720; retained snapshots: 91
Exact duplicate snapshots removed: 0; repeated ticker entries removed: 0


,snapshot_date,valid_from,valid_to
0,2019-12-23,2020-01-02,2020-01-28
1,2020-01-28,2020-01-28,2020-03-03
2,2020-03-03,2020-03-03,2020-04-03
3,2020-04-03,2020-04-03,2020-04-06
4,2020-04-06,2020-04-06,2020-05-12


### 2.2 Creating Membership Intervals

Next, we expand the ticker lists to create one row per ticker and membership interval. A filing qualifies when **`valid_from <= filing_date < valid_to`**. `snapshot_date` preserves the original source date.

We remove incomplete and duplicate ticker/start-date rows, restrict intervals to the study period, shorten overlaps to the next start date, and remove empty intervals.

In [7]:
# Expand the retained snapshots into a tidy ticker-by-interval table.
df_historical_clean = historical_snapshots.assign(
    ticker=historical_snapshots["tickers"].str.split(",")
).explode("ticker", ignore_index=True)
df_historical_clean = df_historical_clean[
    ["ticker", "snapshot_date", "valid_from", "valid_to"]
]

# Remove incomplete rows and repeated ticker/start-date pairs.
df_historical_clean = (
    df_historical_clean.dropna()
    .sort_values(["ticker", "valid_from"])
    .drop_duplicates(["ticker", "valid_from"])
    .reset_index(drop=True)
)

# Keep intervals within the study window and end them by the next interval.
df_historical_clean["valid_from"] = df_historical_clean["valid_from"].clip(lower=study_start)
df_historical_clean["valid_to"] = df_historical_clean["valid_to"].clip(upper=study_end_exclusive)
next_start = df_historical_clean.groupby("ticker")["valid_from"].shift(-1)
df_historical_clean["valid_to"] = df_historical_clean["valid_to"].where(
    next_start.isna() | df_historical_clean["valid_to"].le(next_start), next_start
)

# Remove intervals that have no days remaining.
df_historical_clean = df_historical_clean.loc[
    df_historical_clean["valid_from"].lt(df_historical_clean["valid_to"])
].reset_index(drop=True)

print(f"Cleaned membership rows: {len(df_historical_clean):,}")
print(f"Distinct ticker symbols during the study window: {df_historical_clean['ticker'].nunique():,}")
print("Ticker counts are not necessarily unique company counts: symbols and share classes can change.")
display(df_historical_clean.head(10))


Cleaned membership rows: 45,850
Distinct ticker symbols during the study window: 627
Ticker counts are not necessarily unique company counts: symbols and share classes can change.


,ticker,snapshot_date,valid_from,valid_to
0,A,2019-12-23,2020-01-02,2020-01-28
1,A,2020-01-28,2020-01-28,2020-03-03
2,A,2020-03-03,2020-03-03,2020-04-03
3,A,2020-04-03,2020-04-03,2020-04-06
4,A,2020-04-06,2020-04-06,2020-05-12
5,A,2020-05-12,2020-05-12,2020-05-22
6,A,2020-05-22,2020-05-22,2020-06-22
7,A,2020-06-22,2020-06-22,2020-09-18
8,A,2020-09-18,2020-09-18,2020-09-21
9,A,2020-09-21,2020-09-21,2020-10-07


### 2.3 Combining and Saving Membership Intervals

The previous step created a separate interval for each ticker in each snapshot. We now combine consecutive intervals for the same ticker, keeping separate periods when a company left the index and later returned. The end date remains exclusive: a filing on that date is outside the interval.

The resulting table is saved to `data/processed/historical_membership_clean.csv` for matching filings to historical membership.

In [8]:
# Combine consecutive snapshots into continuous membership periods by ticker.
snapshot_rows = len(df_historical_clean)
previous_end = df_historical_clean.groupby("ticker")["valid_to"].shift()
df_historical_clean["membership_period"] = (
    df_historical_clean["valid_from"].ne(previous_end)
    .groupby(df_historical_clean["ticker"]).cumsum()
)
df_historical_clean = (
    df_historical_clean.groupby(["ticker", "membership_period"], as_index=False)
    .agg(snapshot_date=("snapshot_date", "min"),
         valid_from=("valid_from", "min"),
         valid_to=("valid_to", "max"))
    .drop(columns="membership_period")
)

print(f"Rows before grouping: {snapshot_rows:,}; membership periods: {len(df_historical_clean):,}")
print(f"Distinct ticker symbols: {df_historical_clean['ticker'].nunique():,}")
display(df_historical_clean.head(10))

# Save the cleaned historical membership intervals for reuse.
# processed_dir is set above inside the Phase 3/4 folder.
processed_dir.mkdir(parents=True, exist_ok=True)
df_historical_clean.to_csv(processed_dir / "historical_membership_clean.csv", index=False)
print("Saved data/processed/historical_membership_clean.csv")

Rows before grouping: 45,850; membership periods: 628
Distinct ticker symbols: 627
Saved data/processed/historical_membership_clean.csv


,ticker,snapshot_date,valid_from,valid_to
0,A,2019-12-23,2020-01-02,2026-01-01
1,AAL,2019-12-23,2020-01-02,2024-09-23
2,AAP,2019-12-23,2020-01-02,2023-08-25
3,AAPL,2019-12-23,2020-01-02,2026-01-01
4,ABBV,2019-12-23,2020-01-02,2026-01-01
5,ABC,2019-12-23,2020-01-02,2023-08-30
6,ABMD,2019-12-23,2020-01-02,2022-12-22
7,ABNB,2023-09-18,2023-09-18,2026-01-01
8,ABT,2019-12-23,2020-01-02,2026-01-01
9,ACGL,2022-11-01,2022-11-01,2026-01-01


### 2.4 Loading the Saved Prices

The membership intervals are now saved. Next, we load the daily prices and the list of symbols with missing price data.

During Phase 2, prices were collected for historical index members and SPY with adjustments for splits and dividends. Failed symbols were retried, and results with fewer than 20 trading days were treated as unavailable. Here, we check the date range and available tickers before using the prices to calculate weekly returns.

In [9]:
# Load the saved daily prices and the list of symbols Yahoo could not return.
df_prices_daily = pd.read_csv(raw_dir / "prices" / "daily_stock_prices.csv.gz", parse_dates=["date"])
missing_price_tickers = pd.read_csv(raw_dir / "prices" / "daily_stock_prices_missing.csv")["ticker"].tolist()
print(f"Daily price rows: {len(df_prices_daily):,}; tickers with data: {df_prices_daily['ticker'].nunique()} (including SPY)")
print(f"Date range: {df_prices_daily['date'].min().date()} to {df_prices_daily['date'].max().date()}")
print(f"Symbols with no Yahoo data ({len(missing_price_tickers)}):", missing_price_tickers)
display(df_prices_daily.head())

Daily price rows: 1,058,363; tickers with data: 553 (including SPY)
Date range: 2019-01-02 to 2026-10-19
Symbols with no Yahoo data (75): ['ABC', 'ABMD', 'ADS', 'AGN', 'ALXN', 'ANSS', 'ANTM', 'ARNC', 'ATVI', 'AVB', 'BK', 'BLL', 'CDAY', 'CERN', 'CMA', 'COG', 'CTL', 'CTLT', 'CTRA', 'CTXS', 'CXO', 'DAY', 'DFS', 'DISCA', 'DISCK', 'DISH', 'DRE', 'EA', 'EQR', 'ETFC', 'FBHS', 'FI', 'FLIR', 'FLT', 'FRC', 'GPS', 'HBI', 'HES', 'HFC', 'HOLX', 'IPG', 'JNPR', 'JWN', 'K', 'KSU', 'LEG', 'MMC', 'MRO', 'MXIM', 'MYL', 'NBL', 'NLOK', 'NLSN', 'PBCT', 'PEAK', 'PKI', 'PSKY', 'PXD', 'QRVO', 'RE', 'RTN', 'SEE', 'SIVB', 'TIF', 'TWTR', 'UTX', 'VAR', 'VIAC', 'WBA', 'WBD', 'WCG', 'WLTW', 'WRK', 'XEC', 'XLNX']


,ticker,date,close,volume
0,A,2019-01-02,62.087963,2113300.0
1,A,2019-01-03,59.800667,5383900.0
2,A,2019-01-04,61.870571,3123700.0
3,A,2019-01-07,63.184349,3235100.0
4,A,2019-01-08,64.110641,1578100.0


## 3. Cleaning Insider Transactions

We now turn to the insider transactions. The following steps reduce the raw nonderivative transaction table to Form 4 purchase and sale records with usable dates, shares, and prices. A cleaning log tracks how many rows remain after each filter, while the original source tables stay unchanged.

### 3.1 Adding Filing Details

We join transactions to `SUBMISSION` using `ACCESSION_NUMBER`. This join adds the filing date, company identifier, ticker, and available plan checkbox to each transaction. We pad company CIKs to ten digits so formatting differences do not create separate identifiers. We standardize owner CIKs when adding owner information in Section 3.4.

We retain **Form 4 and amended Form 4/A** filings. Form 3 initial holdings and Form 5 annual reports are outside this disclosure-based sample. The filing date anchors the later event-week assignment; it does not provide the exact public-release time.

In [10]:
# 1a. Merge filing-level fields onto each transaction, then keep Forms 4 and 4/A.
submission_cols = ["ACCESSION_NUMBER", "FILING_DATE", "DOCUMENT_TYPE", "ISSUERCIK",
                   "ISSUERNAME", "ISSUERTRADINGSYMBOL", "AFF10B5ONE"]
trades = df_transactions.merge(df_submission[submission_cols], on="ACCESSION_NUMBER",
                               how="left", validate="many_to_one")
assert trades["FILING_DATE"].notna().all(), "Every transaction should match exactly one filing."
# Some quarters store CIKs without leading zeros (38777 vs 0000038777); pad so one company has one ID.
trades["ISSUERCIK"] = trades["ISSUERCIK"].str.strip().str.zfill(10)

cleaning_log = [{"step": "Raw non-derivative transactions", "rows": len(trades)}]
trades = trades[trades["DOCUMENT_TYPE"].isin(["4", "4/A"])].copy()
cleaning_log.append({"step": "1a. Forms 4 and 4/A only", "rows": len(trades)})
print(trades["DOCUMENT_TYPE"].value_counts().to_string())

DOCUMENT_TYPE
4      1793464
4/A      26865


### 3.2 Selecting Purchases and Sales

After selecting Form 4 and Form 4/A filings, we keep **P (purchase)** and **S (sale)** transaction codes. These include exchange and private transactions. We exclude grants, tax-related share withholding, option exercises, gifts, and other codes because they describe different types of ownership changes. This keeps the purchase/sale definition consistent throughout the analysis.

In [11]:
# 1b. Keep purchase (P) and sale (S) codes, including exchange and private transactions.
print("Transaction codes in Form 4 rows (top 8):")
print(trades["TRANS_CODE"].value_counts().head(8).to_string())
trades = trades[trades["TRANS_CODE"].isin(["P", "S"])].copy()
cleaning_log.append({"step": "1b. Purchase (P) and sale (S) codes only", "rows": len(trades)})

Transaction codes in Form 4 rows (top 8):
TRANS_CODE
S    546649
A    383521
F    296238
M    265436
P    164790
J     49083
G     38632
D     36197


### 3.3 Checking Dates and Trade Values

Next, we convert the transaction-date and filing-date fields into date values. We convert shares, prices, and subsequent holdings into numeric values. We calculate **`trade_value = shares × price`** and **`filing_lag_days = filing_date - trans_date`**, measured in calendar days.

We require positive, observed shares and prices and a filing lag from **0 to 365 days**. Negative lags are inconsistent with the intended transaction-to-filing sequence; the upper limit removes unusually old or potentially miscoded dates. Late filings within that window remain eligible because the analysis follows disclosure. We treat missing validity flags as false so every excluded row is included in the printed removal count. Calendar-day lag alone does not measure compliance with the two-business-day deadline.

In [12]:
# 1c. Parse dates and numbers, compute trade value and filing lag, drop unusable rows.
for col in ["TRANS_DATE", "FILING_DATE"]:
    trades[col] = pd.to_datetime(trades[col], format="%d-%b-%Y", errors="coerce")
for col in ["TRANS_SHARES", "TRANS_PRICEPERSHARE", "SHRS_OWND_FOLWNG_TRANS"]:
    trades[col] = pd.to_numeric(trades[col], errors="coerce")

trades["trade_value"] = trades["TRANS_SHARES"] * trades["TRANS_PRICEPERSHARE"]
trades["filing_lag_days"] = (trades["FILING_DATE"] - trades["TRANS_DATE"]).dt.days

valid = (trades["TRANS_DATE"].notna() & trades["FILING_DATE"].notna()
         & trades["TRANS_SHARES"].gt(0) & trades["TRANS_PRICEPERSHARE"].gt(0)
         & trades["filing_lag_days"].between(0, 365))
valid = valid.fillna(False)  # Missing shares/prices are excluded and counted as removed.
print(f"Rows dropped (bad shares/price, unparseable dates, or lag outside 0-365 days): {(~valid).sum():,}")
trades = trades[valid].copy()
cleaning_log.append({"step": "1c. Valid dates (lag 0-365 days), shares > 0, price > 0", "rows": len(trades)})
print(trades["filing_lag_days"].describe(percentiles=[.5, .9, .99]).round(1).to_string())

Rows dropped (bad shares/price, unparseable dates, or lag outside 0-365 days): 11,419
count    700020.0
mean          4.2
std          20.2
min           0.0
50%           2.0
90%           4.0
99%          78.0
max         365.0


### 3.4 Resolving Amendments

After checking dates and values, we group records by company, owner, transaction date, and transaction code. Within each group, we check for a Form 4/A amendment. If present, we keep the rows from the latest filing and remove earlier versions. Groups without amendments remain unchanged so separate same-day trades are preserved.

For joint filings, we use the lowest owner CIK as a stable grouping identifier. In Section 4.1, we use all reporting owners to assign the highest-ranked role to each filing. These rules are reproducible approximations rather than a manual review of every amendment.

In [13]:
# 1d. Drop superseded rows when a Form 4/A amends an earlier filing.
first_owner = (df_owners.sort_values("RPTOWNERCIK")
               .drop_duplicates("ACCESSION_NUMBER")[["ACCESSION_NUMBER", "RPTOWNERCIK"]])
trades = trades.merge(first_owner, on="ACCESSION_NUMBER", how="left", validate="many_to_one")
trades["RPTOWNERCIK"] = trades["RPTOWNERCIK"].str.strip().str.zfill(10)

group_key = ["ISSUERCIK", "RPTOWNERCIK", "TRANS_DATE", "TRANS_CODE"]
trades["is_amendment"] = trades["DOCUMENT_TYPE"].eq("4/A")
group_has_amendment = trades.groupby(group_key, dropna=False)["is_amendment"].transform("any")

# Latest filing in each group; ties broken by the larger accession number.
latest_accession = (trades.sort_values(["FILING_DATE", "ACCESSION_NUMBER"])
                    .groupby(group_key, dropna=False)["ACCESSION_NUMBER"].transform("last"))
superseded = group_has_amendment & trades["ACCESSION_NUMBER"].ne(latest_accession)
print(f"Groups containing an amendment: {trades.loc[group_has_amendment, group_key].drop_duplicates().shape[0]:,}")
print(f"Superseded rows dropped: {superseded.sum():,}")
trades = trades[~superseded].copy()
cleaning_log.append({"step": "1d. Superseded amendment rows removed", "rows": len(trades)})

Groups containing an amendment: 5,706
Superseded rows dropped: 7,995


### 3.5 Saving the Cleaned Transactions

The transaction filters and amendment checks are complete. We now retain the fields needed for matching and analysis, assign consistent lowercase column names, and standardize the filed ticker symbols. The available plan flag is converted to a numeric field, with missing values left as missing.

After sorting the records by filing date, company identifier, and accession number, we save them to `data/interim/sec_ps_trades.csv.gz`. The cleaning log and table preview below show the result of this stage.

In [14]:
# 1e. Select and rename columns, then save the cleaned trades.
df_trades_clean = trades.rename(columns={
    "ACCESSION_NUMBER": "accession", "ISSUERCIK": "issuer_cik", "ISSUERNAME": "issuer_name",
    "ISSUERTRADINGSYMBOL": "ticker_filed", "RPTOWNERCIK": "owner_cik",
    "TRANS_DATE": "trans_date", "FILING_DATE": "filing_date", "TRANS_CODE": "trans_code",
    "TRANS_SHARES": "shares", "TRANS_PRICEPERSHARE": "price", "SHRS_OWND_FOLWNG_TRANS": "shares_after",
    "SECURITY_TITLE": "security_title", "DOCUMENT_TYPE": "form_type", "SOURCE_QUARTER": "source_quarter",
})[["accession", "issuer_cik", "issuer_name", "ticker_filed", "owner_cik", "trans_date",
    "filing_date", "filing_lag_days", "trans_code", "shares", "price", "trade_value",
    "shares_after", "security_title", "form_type", "AFF10B5ONE", "source_quarter"]]

# Ticker cleanup: trim spaces, uppercase, and treat blanks or "NONE" as missing.
df_trades_clean["ticker_filed"] = (df_trades_clean["ticker_filed"].str.strip().str.upper()
                                   .replace({"": pd.NA, "NONE": pd.NA, "N/A": pd.NA}))
df_trades_clean["plan_10b5_1"] = pd.to_numeric(df_trades_clean.pop("AFF10B5ONE"), errors="coerce")
df_trades_clean = df_trades_clean.sort_values(["filing_date", "issuer_cik", "accession"]).reset_index(drop=True)

# interim_dir is set above inside the Phase 3/4 folder.
interim_dir.mkdir(parents=True, exist_ok=True)
df_trades_clean.to_csv(interim_dir / "sec_ps_trades.csv.gz", index=False)

df_cleaning_log = pd.DataFrame(cleaning_log)
df_cleaning_log["rows_removed"] = -df_cleaning_log["rows"].diff().fillna(0).astype(int)
display(df_cleaning_log)
print(df_trades_clean["trans_code"].value_counts().rename({"P": "purchases", "S": "sales"}).to_string())
print(f"Saved {len(df_trades_clean):,} rows to data/interim/sec_ps_trades.csv.gz")
display(df_trades_clean.head())

trans_code
sales        537365
purchases    154660
Saved 692,025 rows to data/interim/sec_ps_trades.csv.gz


,step,rows,rows_removed
0,Raw non-derivative transactions,1841674,0
1,1a. Forms 4 and 4/A only,1820329,21345
2,1b. Purchase (P) and sale (S) codes only,711439,1108890
3,"1c. Valid dates (lag 0-365 days), shares > 0, ...",700020,11419
4,1d. Superseded amendment rows removed,692025,7995


,accession,issuer_cik,issuer_name,ticker_filed,owner_cik,trans_date,filing_date,filing_lag_days,trans_code,shares,price,trade_value,shares_after,security_title,form_type,source_quarter,plan_10b5_1
0,0001127602-20-000197,0000001750,AAR CORP,AIR,0001027741,2019-12-24,2020-01-02,9,S,4793.0,48.02,230159.86,547399.0,Common Stock,4,2020q1,<NA>
1,0000914190-20-000002,0000007623,ARTS WAY MANUFACTURING CO INC,ARTW,0001022585,2019-12-30,2020-01-02,3,P,1151.0,1.75,2014.25,2056982.0,Common Stock,4,2020q1,<NA>
2,0000914190-20-000002,0000007623,ARTS WAY MANUFACTURING CO INC,ARTW,0001022585,2019-12-31,2020-01-02,2,P,51.0,1.75,89.25,2057033.0,Common Stock,4,2020q1,<NA>
3,0001271163-20-000001,0000009521,BANCROFT FUND LTD,BCV,0001185533,2019-12-30,2020-01-02,3,S,1500.0,26.02,39030.0,150500.0,Common Shares,4,2020q1,<NA>
4,0001271163-20-000001,0000009521,BANCROFT FUND LTD,BCV,0001185533,2019-12-31,2020-01-02,2,S,2500.0,26.21,65525.0,148000.0,Common Shares,4,2020q1,<NA>


## 4. Adding Roles and Index Membership

The cleaned transaction table is ready for matching. We next add insider roles and use the historical membership intervals to retain transactions from companies that belonged to the S&P 500 on the filing date.

### 4.1 Assigning Insider Roles

We use the relationship checkboxes and reported titles in `REPORTINGOWNER` to assign roles in the following order:

| Rank | Role | Rule |
|---|---|---|
| 1 | C-suite | Officer titled CEO, CFO, COO, or President, excluding Vice President. |
| 2 | Other officer | Officer not assigned C-suite rank. |
| 3 | Director | Director not assigned an officer rank. |
| 4 | 10% owner | Large owner not assigned an officer or director rank. |
| 5 | Other | Remaining reported relationships. |

For filings with multiple owners, we assign the highest-ranked role among those owners. We also keep separate director, officer, and large-owner flags to preserve overlapping relationships. This classification supplies the role variable for the planned analysis; it does not directly measure access to information.

In [15]:
# 2a. Build one role record per filing from REPORTINGOWNER.
owners = df_owners[["ACCESSION_NUMBER", "RPTOWNER_RELATIONSHIP", "RPTOWNER_TITLE"]].copy()
relationship = owners["RPTOWNER_RELATIONSHIP"].fillna("")
title = owners["RPTOWNER_TITLE"].fillna("").str.upper()

owners["is_director"] = relationship.str.contains("Director")
owners["is_officer"] = relationship.str.contains("Officer")
owners["is_ten_pct_owner"] = relationship.str.contains("TenPercentOwner")
# C-suite: CEO/CFO/COO by abbreviation or full name, or President that is not Vice President.
names_c_suite = title.str.contains(r"\b(?:CEO|CFO|COO)\b|CHIEF (?:EXECUTIVE|FINANCIAL|OPERATING) OFFICER")
is_president = title.str.contains(r"\bPRESIDENT\b") & ~title.str.contains(r"VICE[\s-]*PRES|\bVP\b")
owners["is_c_suite"] = owners["is_officer"] & (names_c_suite | is_president)

# Rank roles from most senior (1) to least (5); each filing keeps its most senior owner.
owners["role_rank"] = np.select(
    [owners["is_c_suite"], owners["is_officer"], owners["is_director"], owners["is_ten_pct_owner"]],
    [1, 2, 3, 4], default=5)
role_names = {1: "C-suite", 2: "Other officer", 3: "Director", 4: "10% owner", 5: "Other"}
df_roles = (owners.groupby("ACCESSION_NUMBER")
            .agg(role_rank=("role_rank", "min"), is_director=("is_director", "max"),
                 is_officer=("is_officer", "max"), is_ten_pct_owner=("is_ten_pct_owner", "max"),
                 n_owners=("role_rank", "size"), owner_title=("RPTOWNER_TITLE", "first"))
            .reset_index().rename(columns={"ACCESSION_NUMBER": "accession"}))
df_roles["role"] = df_roles["role_rank"].map(role_names)

trades_roles = df_trades_clean.merge(df_roles, on="accession", how="left", validate="many_to_one")
assert trades_roles["role"].notna().all(), "Every filing should have at least one reporting owner."
print(pd.crosstab(trades_roles["role"], trades_roles["trans_code"], margins=True).to_string())

trans_code          P       S     All
role                                 
10% owner       27225   48856   76081
C-suite         44168  196523  240691
Director        64139  115774  179913
Other            4338    4788    9126
Other officer   14790  171424  186214
All            154660  537365  692025


### 4.2 Matching Historical Membership

We match companies to the index **at the filing date**, retaining companies that later left. Because ticker spellings and company identifiers can change, we match the records in three steps:

1. We normalize punctuation, spacing, and multiple symbols in filed ticker fields.
2. We link an index ticker to each company CIK accounting for **at least 10%** of its matched filings during membership. This reduces stray matches while allowing company-ID changes.
3. We join trades by CIK and retain filings within the corresponding membership interval.

For simultaneous index share classes, we keep one record per trade and prefer the ticker matching the filing. We report both linked and unmatched symbols.

In [16]:
# 2b. Link index tickers to SEC company IDs (CIKs), then keep trades filed while the company was an S&P 500 member.
import re

def ticker_tokens(raw):
    """Split a filed ticker field into normalized symbols, e.g. 'LEN, LEN.B' -> ['LEN', 'LENB']."""
    if not isinstance(raw, str) or not raw.strip():
        return []
    text = raw.upper().strip()
    if re.fullmatch(r"[A-Z](?:\s[A-Z])+", text):   # 'N O G' -> 'NOG'
        text = text.replace(" ", "")
    parts = re.split(r"[\s,/_;]+|\bAND\b", text)
    return [re.sub(r"[^A-Z]", "", part) for part in parts if re.sub(r"[^A-Z]", "", part)]

# Pass 1: one row per (filing, normalized symbol); index tickers normalized the same way.
filings = trades_roles[["accession", "issuer_cik", "issuer_name", "ticker_filed", "filing_date"]].drop_duplicates("accession")
filings = filings.assign(symbol=filings["ticker_filed"].map(ticker_tokens)).explode("symbol").dropna(subset=["symbol"])
membership = df_historical_clean.assign(symbol=df_historical_clean["ticker"].str.replace(r"[^A-Z]", "", regex=True))

# Pass 2: count filings per (index ticker, CIK) while the ticker was in the index; keep CIKs with >= 10% share.
candidates = filings.merge(membership[["symbol", "ticker", "valid_from", "valid_to"]], on="symbol")
candidates = candidates[candidates["filing_date"].ge(candidates["valid_from"]) & candidates["filing_date"].lt(candidates["valid_to"])]
links = (candidates.groupby(["ticker", "issuer_cik"])
         .agg(n_filings=("accession", "nunique"), issuer_name=("issuer_name", "first")).reset_index())
links["share"] = links["n_filings"] / links.groupby("ticker")["n_filings"].transform("sum")
dropped_links = links[links["share"].lt(0.10)]
df_ticker_cik = links[links["share"].ge(0.10)]
print(f"Index tickers linked to a CIK: {df_ticker_cik['ticker'].nunique()} of {membership['ticker'].nunique()}")
print(f"Stray (ticker, CIK) links dropped for < 10% share: {len(dropped_links)}")
unlinked = sorted(set(membership["ticker"]) - set(df_ticker_cik["ticker"]))
print("Index tickers with no matching insider filings:", unlinked)

# Pass 3: attach membership intervals by CIK and keep trades whose filing date falls inside one.
cik_intervals = df_ticker_cik[["ticker", "issuer_cik"]].merge(membership[["ticker", "valid_from", "valid_to"]], on="ticker")
sp = trades_roles.merge(cik_intervals.rename(columns={"ticker": "sp_ticker"}), on="issuer_cik", how="inner")
sp = sp[sp["filing_date"].ge(sp["valid_from"]) & sp["filing_date"].lt(sp["valid_to"])]

# A company with two share classes in the index at once (GOOG and GOOGL) would appear twice; keep one row per trade,
# preferring the index ticker that matches what the insider filed.
sp["filed_symbol_match"] = [t in ticker_tokens(f) for t, f in zip(sp["sp_ticker"].str.replace(r"[^A-Z]", "", regex=True), sp["ticker_filed"])]
trade_key = ["accession", "issuer_cik", "owner_cik", "trans_date", "trans_code", "shares", "price", "security_title"]
sp = (sp.sort_values("filed_symbol_match", ascending=False)
        .drop_duplicates(trade_key).drop(columns=["valid_from", "valid_to", "filed_symbol_match"]))
cleaning_log.append({"step": "2b. Company in S&P 500 on filing date", "rows": len(sp)})
print(sp["trans_code"].value_counts().rename({"P": "purchases", "S": "sales"}).to_string())

Index tickers linked to a CIK: 611 of 627
Stray (ticker, CIK) links dropped for < 10% share: 8
Index tickers with no matching insider filings: ['AGN', 'AMTM', 'ARES', 'CPRI', 'CRH', 'CVNA', 'ERIE', 'FIX', 'FOXA', 'FRC', 'NWSA', 'PSKY', 'SBNY', 'SOLS', 'VNT', 'WCG']
trans_code
sales        121168
purchases      6361


### 4.3 Removing Duplicate Trades and Saving the Matches

The transactions have been matched to historical index membership. We now check for repeated records with the same owner, transaction date, code, shares, price, and index ticker. This includes trades reported under related company CIKs.

After removing those duplicates, we save the matched trades to `data/interim/sp500_trades.csv.gz` and the ticker links to `data/interim/sp500_ticker_cik_links.csv`.

In [17]:
# 2c. Drop cross-CIK duplicates, then save the S&P 500 trade file.
dup_across_ciks = sp.duplicated(["owner_cik", "trans_date", "trans_code", "shares", "price", "sp_ticker"])
print(f"Duplicate trades reported under a second CIK: {dup_across_ciks.sum():,}")
df_sp_trades = sp[~dup_across_ciks].sort_values(["filing_date", "sp_ticker", "accession"]).reset_index(drop=True)
cleaning_log.append({"step": "2c. Same trade under two CIKs removed", "rows": len(df_sp_trades)})

df_sp_trades.to_csv(interim_dir / "sp500_trades.csv.gz", index=False)
df_ticker_cik.to_csv(interim_dir / "sp500_ticker_cik_links.csv", index=False)
print(f"Saved {len(df_sp_trades):,} trades in {df_sp_trades['issuer_cik'].nunique()} companies to data/interim/sp500_trades.csv.gz")
print(pd.crosstab(df_sp_trades["role"], df_sp_trades["trans_code"], margins=True).to_string())

Duplicate trades reported under a second CIK: 161
Saved 127,368 trades in 593 companies to data/interim/sp500_trades.csv.gz
trans_code        P       S     All
role                               
10% owner       742    3291    4033
C-suite        1205   51938   53143
Director       3633   23605   27238
Other            20     542     562
Other officer   758   41634   42392
All            6358  121010  127368


## 5. Preparing Weekly Returns

With the eligible trades saved, we return to the daily price data. This stage checks the prices, calculates Friday-to-Friday returns for stocks and SPY, and selects an available price history for each company. These returns will be used to measure stock performance around the disclosure events.

### 5.1 Calculating Weekly Returns

We first remove repeated ticker-day records and rows with missing or nonpositive closing prices. For each ticker, the code takes the last available close in each week ending Friday and calculates **current close / previous close - 1**. If Friday is a market holiday, the last available close may be Thursday's.

The same calculation gives us SPY's weekly returns. We join those market returns to the stock returns by week.

In [18]:
# 3a. Clean daily prices, then compute Friday-to-Friday weekly returns for every ticker and for SPY.
prices = df_prices_daily.drop_duplicates(["ticker", "date"])
bad_prices = prices["close"].le(0) | prices["close"].isna()
print(f"Duplicate ticker-days removed: {len(df_prices_daily) - len(prices):,}; non-positive or missing closes removed: {bad_prices.sum():,}")
prices = prices[~bad_prices]

# Last close of each week ending Friday ('W-FRI'), then the percentage change from the prior week.
weekly_close = (prices.set_index("date").groupby("ticker")["close"]
                .resample("W-FRI").last().dropna().reset_index())
weekly_close["ret"] = weekly_close.groupby("ticker")["close"].pct_change()
df_weekly = weekly_close.dropna(subset=["ret"]).rename(columns={"date": "week"})

market = df_weekly.loc[df_weekly["ticker"].eq("SPY"), ["week", "ret"]].rename(columns={"ret": "mkt_ret"})
df_weekly = df_weekly[df_weekly["ticker"].ne("SPY")].merge(market, on="week", how="inner")
print(f"Weekly return rows: {len(df_weekly):,} across {df_weekly['ticker'].nunique()} stocks; "
      f"{market['week'].min().date()} to {market['week'].max().date()}")
print("Weekly stock returns, percent:")
print((df_weekly["ret"] * 100).describe(percentiles=[.01, .5, .99]).round(2).to_string())

Duplicate ticker-days removed: 0; non-positive or missing closes removed: 0
Weekly return rows: 219,378 across 552 stocks; 2019-01-11 to 2026-10-23
Weekly stock returns, percent:
count    219378.00
mean          0.34
std           5.21
min         -61.01
1%          -13.87
50%           0.32
99%          15.20
max         129.20


### 5.2 Linking Company Price Histories

Weekly returns are now available by ticker. To link them to the transactions, we select one price history for each company CIK. Among linked tickers with available prices, we choose the one with the latest membership end date.

This rule allows an older symbol, such as FB, to use an available price history under a newer symbol, such as META. The output also lists companies without a usable price history, since their events cannot receive return estimates.

In [19]:
# 3b. For each company (CIK), pick the linked index ticker with price data and the latest membership end date.
tickers_with_prices = set(df_weekly["ticker"])
latest_end = df_historical_clean.groupby("ticker")["valid_to"].max().rename("latest_end").reset_index()
price_choice = (df_ticker_cik[["issuer_cik", "ticker"]].merge(latest_end, on="ticker")
                .loc[lambda d: d["ticker"].isin(tickers_with_prices)]
                .sort_values("latest_end").drop_duplicates("issuer_cik", keep="last")
                .rename(columns={"ticker": "price_ticker"})[["issuer_cik", "price_ticker"]])

renamed = (df_sp_trades[["issuer_cik", "sp_ticker"]].drop_duplicates()
           .merge(price_choice, on="issuer_cik").query("sp_ticker != price_ticker"))
print(f"Companies with a usable price series: {price_choice['issuer_cik'].nunique()} of {df_sp_trades['issuer_cik'].nunique()}")
print("Examples where an old index ticker uses its newer ticker's prices:")
print(renamed.drop_duplicates("sp_ticker").head(10).to_string(index=False))

no_prices = df_sp_trades[~df_sp_trades["issuer_cik"].isin(price_choice["issuer_cik"])]
print(f"\nTrades in companies with no price series: {len(no_prices):,} "
      f"({no_prices['trans_code'].eq('P').sum():,} purchases) in {no_prices['issuer_cik'].nunique()} companies, e.g.",
      sorted(no_prices["sp_ticker"].unique())[:12])

Companies with a usable price series: 540 of 593
Examples where an old index ticker uses its newer ticker's prices:
issuer_cik sp_ticker price_ticker
0001326801        FB         META
0001140859       ABC          COR
0000009389       BLL         BALL
0001156039      ANTM          ELV
0001095073        RE           EG
0000031791       PKI         RVTY
0000101829       UTX          RTX
0001175454       FLT         CPAY
0000765880      PEAK          DOC
0000813828      VIAC         PARA

Trades in companies with no price series: 4,511 (271 purchases) in 53 companies, e.g. ['ABMD', 'ADS', 'ALXN', 'ANSS', 'ATVI', 'AVB', 'BK', 'CDAY', 'CERN', 'CMA', 'COG', 'CTLT']


## 6. Building the Event Dataset

We now have the matched transactions and weekly returns needed to construct the event dataset. Each event represents **one company, one event week, and one purchase/sale direction**. After grouping the trades, we estimate expected returns from market movements and calculate how actual returns differ from those expectations.

### 6.1 Combining Trades into Events

Grouping trades prevents several transactions at one company in the same week from being counted as separate signals. We assign each filing to the first Friday on or after **filing date plus one weekday** and use that week's close as the entry price. This is an assumed timing rule; it does not account for exact filing times or market holidays.

For each event, we count trades and distinct retained owners, sum purchase or sale dollars, and take the natural log of that total. We retain the highest-ranked role, whether any trade has C-suite rank, the maximum observed plan flag, average filing lag, and whether the company has both purchase and sale events that week.

In [20]:
# 4a. Assign each trade to its event week and aggregate to one row per company, week, and direction.
df_sp_trades["public_date"] = df_sp_trades["filing_date"] + pd.offsets.BDay(1)
# First Friday on or after the public date (rollforward leaves a date that is already a Friday unchanged).
df_sp_trades["event_week"] = df_sp_trades["public_date"].map(pd.offsets.Week(weekday=4).rollforward)

df_events = (df_sp_trades.groupby(["issuer_cik", "event_week", "trans_code"])
             .agg(issuer_name=("issuer_name", "first"), sp_ticker=("sp_ticker", "first"),
                  n_trades=("accession", "size"), n_insiders=("owner_cik", "nunique"),
                  total_value=("trade_value", "sum"), role_rank=("role_rank", "min"),
                  any_c_suite=("role_rank", lambda r: int((r == 1).any())),
                  any_plan_10b5_1=("plan_10b5_1", "max"), mean_filing_lag=("filing_lag_days", "mean"),
                  first_filing=("filing_date", "min"))
             .reset_index())
df_events["role"] = df_events["role_rank"].map(role_names)
df_events["log_value"] = np.log(df_events["total_value"])
both = df_events.groupby(["issuer_cik", "event_week"])["trans_code"].transform("nunique")
df_events["mixed_week"] = both.gt(1).astype(int)
df_events = df_events.merge(price_choice, on="issuer_cik", how="left")
print(df_events.groupby("trans_code").agg(events=("issuer_cik", "size"), companies=("issuer_cik", "nunique"),
                                          trades=("n_trades", "sum")).rename(index={"P": "purchases", "S": "sales"}).to_string())

            events  companies  trades
trans_code                           
purchases     2356        459    6358
sales        25464        582  121010


### 6.2 Estimating Expected Returns

With the events defined, we estimate each stock's relationship with SPY using up to **52 available weekly returns before the event**, requiring at least **40 observations**:

`stock return = alpha + beta × SPY return + residual`

Here, alpha is the fitted intercept and beta is the stock's response to market movements. Using only earlier returns avoids fitting the benchmark to the outcome being measured. We calculate the single-variable regression slope as covariance divided by market-return variance, then calculate its intercept from the two means.

In [21]:
# 4b. Estimate alpha and beta for each event from the 52 weeks before its event week.
weekly_by_ticker = {t: g.set_index("week")[["ret", "mkt_ret"]].sort_index() for t, g in df_weekly.groupby("ticker")}
ESTIMATION_WEEKS, MIN_WEEKS = 52, 40

def market_model(ticker, event_week):
    """Return (alpha, beta, weeks used) from the 52 weeks strictly before event_week."""
    series = weekly_by_ticker.get(ticker)
    if series is None:
        return np.nan, np.nan, 0
    window = series.loc[:event_week - pd.Timedelta(days=1)].tail(ESTIMATION_WEEKS)
    if len(window) < MIN_WEEKS:
        return np.nan, np.nan, len(window)
    beta = np.cov(window["ret"], window["mkt_ret"])[0, 1] / window["mkt_ret"].var()
    alpha = window["ret"].mean() - beta * window["mkt_ret"].mean()
    return alpha, beta, len(window)

fits = [market_model(t, w) for t, w in zip(df_events["price_ticker"], df_events["event_week"])]
df_events[["alpha", "beta", "estimation_weeks"]] = pd.DataFrame(fits, index=df_events.index)
print(f"Events with an estimated beta: {df_events['beta'].notna().sum():,} of {len(df_events):,}")
print(df_events["beta"].describe(percentiles=[.05, .5, .95]).round(2).to_string())

Events with an estimated beta: 26,100 of 27,820
count    26100.00
mean         1.02
std          0.50
min         -1.29
5%           0.25
50%          1.00
95%          1.89
max          4.75


### 6.3 Calculating Abnormal Returns

Next, we use the estimated alpha and beta to calculate each week's abnormal return after entry: **abnormal return = actual return - (alpha + beta × SPY return)**. We sum these differences over **1, 4, 13, and 26 weeks** to obtain cumulative abnormal returns (CARs). Values remain missing when the required price or estimation history is unavailable. Stored returns are decimals; multiplying by 100 expresses them in percentage points.

We calculate `car_pre4` over the four weeks ending with the event week to examine recent price movement. Because this includes the event week, it can include a response to the disclosure and is not strictly a pre-disclosure measure.

In [22]:
# 4c. Abnormal returns: sum (actual - expected) over 1, 4, 13, 26 weeks after the event week, plus the 4 weeks ending with it.
HORIZONS = [1, 4, 13, 26]

def cumulative_abnormal_returns(ticker, event_week, alpha, beta):
    series = weekly_by_ticker.get(ticker)
    out = {f"car_{h}w": np.nan for h in HORIZONS} | {"car_pre4": np.nan}
    if series is None or np.isnan(beta):
        return out
    abnormal = series["ret"] - (alpha + beta * series["mkt_ret"])
    after = abnormal.loc[event_week + pd.Timedelta(days=1):]
    for h in HORIZONS:
        window = after.iloc[:h]
        # Require h consecutive weekly returns that end within h weeks plus a holiday buffer of the event week.
        if len(window) == h and (window.index[-1] - event_week).days <= 7 * h + 7:
            out[f"car_{h}w"] = window.sum()
    before = abnormal.loc[:event_week].tail(4)
    if len(before) == 4:
        out["car_pre4"] = before.sum()
    return out

cars = [cumulative_abnormal_returns(t, w, a, b) for t, w, a, b in
        zip(df_events["price_ticker"], df_events["event_week"], df_events["alpha"], df_events["beta"])]
df_events = pd.concat([df_events, pd.DataFrame(cars, index=df_events.index)], axis=1)
car_cols = [f"car_{h}w" for h in HORIZONS]
print("Events with each CAR available:")
print(df_events.groupby("trans_code")[car_cols + ["car_pre4"]].count().rename(index={"P": "purchases", "S": "sales"}).to_string())

Events with each CAR available:
            car_1w  car_4w  car_13w  car_26w  car_pre4
trans_code                                            
purchases     2227    2227     2227     2227      2227
sales        23873   23873    23873    23873     23873


### 6.4 Saving the Event Dataset

The return measures have now been calculated. This cell puts the event columns in a consistent order and saves the table to `data/processed/insider_events.csv`.

Events with missing prices or beta remain in the file so that we can report data coverage. They cannot enter analyses that require those return measures. The output below shows the complete cleaning log and a preview of purchase events.

In [23]:
# 4d. Order columns, save the analysis-ready event table, and print the full cleaning funnel.
event_cols = ["issuer_cik", "issuer_name", "sp_ticker", "price_ticker", "trans_code", "event_week", "first_filing",
              "n_trades", "n_insiders", "total_value", "log_value", "role", "role_rank", "any_c_suite",
              "any_plan_10b5_1", "mixed_week", "mean_filing_lag", "alpha", "beta", "estimation_weeks",
              "car_pre4"] + car_cols
df_events = df_events[event_cols].sort_values(["event_week", "issuer_cik", "trans_code"]).reset_index(drop=True)
df_events.to_csv(processed_dir / "insider_events.csv", index=False)

has_returns = df_events["beta"].notna()
cleaning_log.append({"step": "4a. Trades rolled up into company-week events (rows = events)", "rows": len(df_events)})
cleaning_log.append({"step": "4b. Events with a price series and >= 40 weeks for beta", "rows": int(has_returns.sum())})
df_cleaning_log = pd.DataFrame(cleaning_log)
display(df_cleaning_log)
print(f"Saved {len(df_events):,} events ({df_events['trans_code'].eq('P').sum():,} purchase events, "
      f"{df_events['trans_code'].eq('S').sum():,} sale events) to data/processed/insider_events.csv")
display(df_events[df_events["trans_code"].eq("P")].head())

Saved 27,820 events (2,356 purchase events, 25,464 sale events) to data/processed/insider_events.csv


,step,rows
0,Raw non-derivative transactions,1841674
1,1a. Forms 4 and 4/A only,1820329
2,1b. Purchase (P) and sale (S) codes only,711439
3,"1c. Valid dates (lag 0-365 days), shares > 0, ...",700020
4,1d. Superseded amendment rows removed,692025
5,2b. Company in S&P 500 on filing date,127529
6,2c. Same trade under two CIKs removed,127368
7,4a. Trades rolled up into company-week events ...,27820
8,4b. Events with a price series and >= 40 weeks...,26100


,issuer_cik,issuer_name,sp_ticker,price_ticker,trans_code,event_week,first_filing,n_trades,n_insiders,total_value,log_value,role,role_rank,any_c_suite,any_plan_10b5_1,mixed_week,mean_filing_lag,alpha,beta,estimation_weeks,car_pre4,car_1w,car_4w,car_13w,car_26w
1,0000051253,INTERNATIONAL FLAVORS & FRAGRANCES INC,IFF,IFF,P,2020-01-03,2020-01-02,2,1,8272210.0,15.928412,10% owner,4,0,<NA>,0,3.0,-0.005327,1.019928,51,-0.112750,-0.007962,0.062931,0.090631,0.209370
5,0000940944,DARDEN RESTAURANTS INC,DRI,DRI,P,2020-01-03,2020-01-02,1,1,163110.0,12.00218,Director,3,0,<NA>,0,2.0,-0.001759,0.746388,51,-0.075855,0.021369,0.052300,-0.443717,-0.010890
21,0000023217,CONAGRA BRANDS INC.,CAG,CAG,P,2020-01-10,2020-01-03,2,1,1359489.0,14.122619,Director,3,0,<NA>,1,2.0,0.003111,1.340501,52,0.086221,0.018333,-0.027635,0.173048,0.069355
27,0000051253,INTERNATIONAL FLAVORS & FRAGRANCES INC,IFF,IFF,P,2020-01-10,2020-01-03,1,1,8729840.0,15.982258,10% owner,4,0,<NA>,0,3.0,-0.005349,1.020515,52,-0.061284,0.056359,0.110913,0.268824,0.192880
55,0000822416,PULTEGROUP INC/MI/,PHM,PHM,P,2020-01-10,2020-01-07,1,1,14909.034,9.609723,Director,3,0,<NA>,0,4.0,0.006139,0.285479,52,-0.039494,0.033403,0.114467,-0.206934,-0.000015


### 6.5 Reviewing the Exported Datasets

At this point, we have saved the following files under `insider-trading-phase3-4`. These tables record the intermediate steps and the final event dataset.

| File | Contents |
|---|---|
| `data/interim/sec_ps_trades.csv.gz` | Cleaned SEC purchase and sale transactions. |
| `data/interim/sp500_trades.csv.gz` | Transactions matched to historical S&P 500 membership. |
| `data/interim/sp500_ticker_cik_links.csv` | Ticker-to-company links. |
| `data/processed/historical_membership_clean.csv` | Cleaned membership intervals. |
| `data/processed/insider_events.csv` | Final company-week-direction events. |

Next, we check the event export and save the supporting weekly returns and cleaning log.

## 7. Checking the Final Export

The event table has been exported. As a final check, we confirm that each row has a unique combination of company, event week, and purchase/sale direction. We then reload `insider_events.csv` and compare it with the table in memory to check that the export preserved the values.

When reading the CSV, we keep CIKs as text to preserve leading zeros, parse the date columns, and use an integer type that allows missing values for the plan flag. We also save `weekly_stock_prices.csv` and `cleaning_log.csv` in `data/processed` for use in the analysis notebook. Events with missing returns remain available for reporting coverage and defining the analysis sample.

In [24]:
# Verify event keys and CSV round-trip, and save supporting analysis inputs.
assert not df_events.duplicated(["issuer_cik", "event_week", "trans_code"]).any()
reloaded_events = pd.read_csv(processed_dir / "insider_events.csv", dtype={"issuer_cik": "string", "any_plan_10b5_1": "Int64"},
                              parse_dates=["event_week", "first_filing"])
pd.testing.assert_frame_equal(df_events, reloaded_events, check_dtype=False, check_exact=False)
df_weekly.to_csv(processed_dir / "weekly_stock_prices.csv", index=False)
df_cleaning_log.to_csv(processed_dir / "cleaning_log.csv", index=False)
print(f"Verified {len(reloaded_events):,} events and {len(reloaded_events.columns)} columns.")
print("Saved weekly_stock_prices.csv and cleaning_log.csv in", processed_dir)


Verified 27,820 events and 25 columns.
Saved weekly_stock_prices.csv and cleaning_log.csv in C:\Users\vansh\.codex\.chatgpt-projects\g-p-6ad697e67f448191ba13f713a97c3142\step4-run\DSproject-phase2-part1-sec-cleaning\insider-trading-phase3-4\data\processed


## 8. Moving from Cleaning to Analysis

The cleaning steps and export checks are complete. We now have the cleaned CSV files needed for `phase3_insider_trading.ipynb`.

Next, we will move to that notebook and load `data/processed/insider_events.csv` using the column types shown above. The weekly returns and cleaning log are available as supporting files. The research models, significance tests, figures, and conclusions will be developed in the analysis notebook.

### Limitations Carried into the Analysis

The cleaning process retains the Phase 2 timing and matching rules. The one-weekday filing buffer does not account for exchange holidays. Weekly return calculations use available observations, and the CAR endpoint check permits a one-week buffer. Ticker matching, missing price histories, amendment handling, and overlapping events also affect how we interpret the final sample.